# A multimodal dataset of cardiovascular and respiratory time series for the study of autonomic responses to orthostatic and mental stress Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and explore a Croissant-described dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.s955-3wd4/fair2.json`

### Important note about identifiers
Throughout this notebook, dataset entities should be referenced by their `@id` values whenever possible. This includes record sets, fields, columns, and related metadata entities. Because this dataset may expose record-set information differently depending on the runtime and resolver, the notebook first inspects the metadata dynamically and then uses discovered `@id` values for downstream steps.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`, then inspect the high-level dataset description.

This section intentionally accesses `dataset.metadata` as a single object and avoids treating it like a subscriptable dictionary-like structure.

In [ ]:
import json
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.s955-3wd4/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset loaded successfully.')
print(metadata)

In [ ]:
# Convert metadata to JSON-serializable form for inspection only.
# We keep `metadata` itself as an object and do not subscript it directly.
metadata_json = metadata.to_json()

dataset_name = metadata_json.get('name', 'Unknown dataset')
dataset_description = metadata_json.get('description', 'No description available.')
dataset_id = metadata_json.get('@id', 'No @id found')
dataset_version = metadata_json.get('version', 'Unknown')
dataset_license = metadata_json.get('license', 'Unknown')

print(f'Name: {dataset_name}')
print(f'Dataset @id: {dataset_id}')
print(f'Version: {dataset_version}')
print(f'License: {dataset_license}')
print(f'Description: {dataset_description}')

## 2. Data Overview
Review the available metadata, focusing on record sets and fields.

The FAIR^2 package summary provided with the prompt indicates dataset-level metadata but does not explicitly list record sets. In practice, Croissant datasets may expose record sets through the metadata object, through `to_json()`, or via resolvable file objects. The next cells inspect the structure safely and dynamically.

In [ ]:
# Inspect top-level metadata keys
print('Top-level metadata keys:')
print(sorted(metadata_json.keys()))

In [ ]:
# Discover record sets from the JSON representation, if present
record_sets_metadata = metadata_json.get('recordSet', []) or []

print(f'Number of record sets declared in metadata_json["recordSet"]: {len(record_sets_metadata)}')

if record_sets_metadata:
    overview_rows = []
    for rs in record_sets_metadata:
        rs_id = rs.get('@id', 'UNKNOWN_RECORD_SET_ID')
        rs_name = rs.get('name', '')
        rs_desc = rs.get('description', '')
        field_count = len(rs.get('field', []) or [])
        overview_rows.append({
            'record_set_@id': rs_id,
            'record_set_name': rs_name,
            'field_count': field_count,
            'description': rs_desc
        })
    display(pd.DataFrame(overview_rows))
else:
    print('No record sets are explicitly listed in the metadata JSON. This can happen when the schema emphasizes dataset-level FAIR metadata or when record-level access is provided through distributions.')

In [ ]:
# If record sets exist, inspect fields using their @id values.
if record_sets_metadata:
    first_record_set = record_sets_metadata[0]
    first_record_set_id = first_record_set.get('@id')
    print(f'First record set @id: {first_record_set_id}')
    
    field_rows = []
    for field in first_record_set.get('field', []) or []:
        field_rows.append({
            'field_@id': field.get('@id', 'UNKNOWN_FIELD_ID'),
            'name': field.get('name', ''),
            'dataType': field.get('dataType', ''),
            'description': field.get('description', ''),
            'source': field.get('source', '')
        })
    display(pd.DataFrame(field_rows))
else:
    first_record_set_id = None
    print('Skipping field inspection because no record sets were declared.')

### Trying to read records directly
If record sets are available, we can iterate through their records using the record set `@id`.

The cell below follows the template style while remaining robust if no record sets are exposed.

In [ ]:
if first_record_set_id is not None:
    print(f'Previewing records from record set: {first_record_set_id}')
    for i, x in enumerate(dataset.records(record_set=first_record_set_id)):
        print(x)
        if i >= 2:
            break
else:
    print('No record set @id is available for direct record iteration.')

## 3. Data Extraction
Load data from available record sets into pandas DataFrames.

This section uses variables for dynamic handling of record set paths and field identifiers, following the requested style. If no record sets are available in the Croissant metadata, the notebook falls back to inspecting declared distributions so you can still understand how the dataset is packaged.

In [ ]:
dataframes = {}
record_set_ids = [rs.get('@id') for rs in record_sets_metadata if rs.get('@id')]

print('Discovered record set @id values:')
print(record_set_ids)

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f'Loaded {len(df)} rows from record set {record_set_id}')
    except Exception as e:
        print(f'Could not load records for {record_set_id}: {e}')

print('\nAvailable DataFrames:')
print(list(dataframes.keys()))

In [ ]:
# Show a quick preview of the first successfully loaded DataFrame, if any
if dataframes:
    example_record_set_id = next(iter(dataframes))
    print(f'Example record set @id: {example_record_set_id}')
    print('Columns:')
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print('No DataFrames were loaded from record sets.')
    
    distributions = metadata_json.get('distribution', []) or []
    if distributions:
        dist_df = pd.DataFrame([
            {
                'distribution_@id': d.get('@id', 'UNKNOWN_DISTRIBUTION_ID'),
                'name': d.get('name', ''),
                'contentUrl': d.get('contentUrl', ''),
                'encodingFormat': d.get('encodingFormat', ''),
                'description': d.get('description', '')
            }
            for d in distributions
        ])
        print('Declared distributions:')
        display(dist_df)
    else:
        print('No distributions were found in metadata_json either.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as:
- identifying numeric fields,
- filtering records based on a threshold,
- normalizing a numeric field,
- grouping by a categorical or identifier-like field.

Because Croissant datasets can vary in structure, the analysis below chooses fields dynamically from the loaded record set.

In [ ]:
if dataframes:
    record_set_id = next(iter(dataframes))
    analysis_df = dataframes[record_set_id].copy()
    
    print(f'Using record set @id for analysis: {record_set_id}')
    print(f'Shape: {analysis_df.shape}')
    
    numeric_columns = analysis_df.select_dtypes(include=[np.number]).columns.tolist()
    object_columns = analysis_df.select_dtypes(include=['object', 'string']).columns.tolist()
    
    print('Numeric columns discovered:')
    print(numeric_columns)
    print('\nNon-numeric columns discovered:')
    print(object_columns[:20])
else:
    print('EDA cannot proceed because no tabular record set was loaded.')

In [ ]:
if dataframes:
    analysis_df = dataframes[record_set_id].copy()
    numeric_columns = analysis_df.select_dtypes(include=[np.number]).columns.tolist()
    object_columns = analysis_df.select_dtypes(include=['object', 'string']).columns.tolist()
    
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        threshold = analysis_df[numeric_field_id].median()
        filtered_df = analysis_df[analysis_df[numeric_field_id] > threshold].copy()
        
        print(f'Chosen numeric field: {numeric_field_id}')
        print(f'Filtering threshold (median): {threshold}')
        print(f'Filtered records with {numeric_field_id} > {threshold}:')
        display(filtered_df.head())
        
        std_value = filtered_df[numeric_field_id].std()
        if pd.notna(std_value) and std_value != 0:
            normalized_col = f'{numeric_field_id}_normalized'
            filtered_df[normalized_col] = (
                filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
            ) / std_value
            print(f'Normalized field created: {normalized_col}')
            display(filtered_df[[numeric_field_id, normalized_col]].head())
        else:
            print(f'Could not normalize {numeric_field_id} because its standard deviation is zero or undefined after filtering.')
        
        group_field_id = None
        for candidate in object_columns:
            if analysis_df[candidate].nunique(dropna=True) > 1 and analysis_df[candidate].nunique(dropna=True) <= 20:
                group_field_id = candidate
                break
        
        if group_field_id is not None:
            print(f'Grouping by field: {group_field_id}')
            grouped_df = filtered_df.groupby(group_field_id)[[numeric_field_id]].mean(numeric_only=True)
            display(grouped_df.head())
        else:
            print('No suitable grouping field was found automatically.')
    else:
        print('No numeric columns are available for thresholding and normalization.')

In [ ]:
# Optional outlier handling example using IQR on the chosen numeric field
if dataframes and numeric_columns:
    analysis_df = dataframes[record_set_id].copy()
    numeric_field_id = numeric_columns[0]
    
    q1 = analysis_df[numeric_field_id].quantile(0.25)
    q3 = analysis_df[numeric_field_id].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    no_outliers_df = analysis_df[
        (analysis_df[numeric_field_id] >= lower_bound) &
        (analysis_df[numeric_field_id] <= upper_bound)
    ].copy()
    
    print(f'Outlier handling for field: {numeric_field_id}')
    print(f'Lower bound: {lower_bound}')
    print(f'Upper bound: {upper_bound}')
    print(f'Original rows: {len(analysis_df)}')
    print(f'Rows after removing IQR outliers: {len(no_outliers_df)}')

## 5. Visualization
Visualize numeric distributions and, where possible, relationships between fields.

The plots are generated dynamically from the first loaded record set.

In [ ]:
if dataframes and numeric_columns:
    analysis_df = dataframes[record_set_id].copy()
    numeric_field_id = numeric_columns[0]
    
    plt.figure(figsize=(10, 5))
    sns.histplot(analysis_df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
else:
    print('No numeric data available for visualization.')

In [ ]:
if dataframes and len(numeric_columns) >= 2:
    analysis_df = dataframes[record_set_id].copy()
    x_field_id = numeric_columns[0]
    y_field_id = numeric_columns[1]
    
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=analysis_df, x=x_field_id, y=y_field_id, alpha=0.6)
    plt.title(f'{y_field_id} vs {x_field_id}')
    plt.xlabel(x_field_id)
    plt.ylabel(y_field_id)
    plt.show()
else:
    print('At least two numeric columns are required for a scatter plot.')

In [ ]:
if dataframes and numeric_columns:
    analysis_df = dataframes[record_set_id].copy()
    corr_df = analysis_df[numeric_columns].corr(numeric_only=True)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_df, cmap='coolwarm', center=0)
    plt.title(f'Correlation heatmap for numeric fields in {record_set_id}')
    plt.show()
else:
    print('No numeric columns available for a correlation heatmap.')

## 6. Conclusion
This notebook showed how to:
- load a Croissant dataset with `mlcroissant`,
- inspect dataset metadata safely,
- discover record sets and their field `@id` values when available,
- extract records into pandas DataFrames,
- apply simple EDA steps such as filtering, normalization, grouping, and outlier handling,
- create basic visualizations for numeric data.

For this FAIR^2 package, the available metadata strongly emphasizes rich dataset-level documentation about cardiovascular and respiratory time series collected across protocol phases such as REST, HUT, REST-2, MA, and REST-3. If record sets are not directly exposed through the resolved Croissant document, the metadata and distribution inspection steps still provide a reproducible starting point for downstream access and analysis.